# Moscow Land Cover Analysis — Colab runner

Тонкий управляющий ноутбук: **Runtime → Run all**. Вся логика — в пакете
`moscow_landcover`; здесь только окружение Colab и вызовы пайплайна.
Диагностика (F1/F2) и экспорт — закомментированы, раскомментируйте при перепроверке.


In [ ]:
# ==== 1. Google Drive ====
from google.colab import drive  # noqa: F401
drive.mount('/content/drive')


In [ ]:
# ==== 2. Установка пакета из git + проверка ревизии ====
# Репозиторий публичный — авторизация не нужна.
# Для воспроизводимого (эталонного) прогона задайте в MLC_REF тег или commit hash.
MLC_REF = 'main'
MLC_URL = 'https://github.com/maxlansky1/moscow-landcover-analysis.git'

# Сносим возможную старую editable-копию, затем ставим из git.
# --upgrade: pip иначе НЕ переустановит VCS-пакет с той же версией (pip #7678).
!pip uninstall -y -q moscow-landcover
!pip install -q --upgrade "moscow_landcover @ git+{MLC_URL}@{MLC_REF}"

import importlib.metadata as md
import json

import moscow_landcover

# PEP 610: direct_url.json хранит точный commit_id установленного кода.
commit = '?'
try:
    direct = json.loads(md.distribution('moscow_landcover').read_text('direct_url.json') or '{}')
    commit = direct.get('vcs_info', {}).get('commit_id', '?')
except Exception as exc:  # noqa: BLE001
    print('direct_url.json недоступен:', exc)

print('moscow_landcover', moscow_landcover.__version__, '| ref', MLC_REF)
print('import     :', moscow_landcover.__file__)
print('pkg commit :', commit)


In [ ]:
# ==== 3. Переменные окружения (не секреты в коде) ====
import os

os.environ['MLC_PROJECT_PATH'] = '/content/drive/MyDrive/Moscow_LandCover_Analysis'
os.environ['MLC_EE_PROJECT'] = 'moscow-landcover-analysis'


In [ ]:
# ==== 4. Конфигурация, пути, логирование, проверка границ ====
from pathlib import Path

from moscow_landcover.config import load_config, paths_from_config
from moscow_landcover.logging import start_run_log

cfg = load_config()
paths = paths_from_config(cfg, create=True)
start_run_log(paths, label='setup')

print('BASE       :', cfg.base_path)
print('GEE project:', cfg.project_id)
for name in ('moscow_historical.geojson', 'moscow_greater.geojson'):
    p = Path(paths['BOUNDARIES']) / name
    print(p, '✅' if p.exists() else '❌ ОТСУТСТВУЕТ')

In [ ]:
# ==== 5. Аутентификация и проверка Earth Engine ====
start_run_log(paths, label='gee')

from moscow_landcover import gee

gee.authenticate(auth_mode='colab')   # откроется ссылка авторизации
gee.initialize()                      # проект берётся из MLC_EE_PROJECT
gee.check()                           # сервер + коллекция + тестовый снимок

## Smoke-тест: 1 территория × 1 год

Полный цикл на одном годе (композит → ESA-точки → RF → OA/Kappa → площади).


In [ ]:
start_run_log(paths, label='smoke')

from moscow_landcover.pipeline import run

smoke_results, smoke_failed = run(cfg, plan=[('Historical', 2026)])
print('failed:', smoke_failed)

## Полный прогон: 5 (Historical) + 3 (Greater)

Результаты → `03_Intermediate/results_by_year.json`,
лог → `04_Logs/<дата>/run_*.log`.

In [ ]:
start_run_log(paths, label='full')

from moscow_landcover.pipeline import run

results, failed = run(cfg)
print(f'прогонов: {sum(len(v) for v in results.values())}, провалов: {len(failed)}')

## Итоговая сводка (H2)

In [ ]:
start_run_log(paths, label='report')

from moscow_landcover.report import summary

summary(cfg, results)

## Эталон (reference): сверка на воспроизводимость

`compare` сверяет прогон с зафиксированным эталоном. Если эталона ещё нет —
один раз выполните ячейку ниже `freeze_reference` (раскомментировав её).


In [ ]:
start_run_log(paths, label='compare')

from moscow_landcover import reference

ref = reference.load_reference(reference.default_reference_path(paths))
if ref is None:
    print('Эталон не найден. Раскомментируйте ячейку ниже и зафиксируйте его.')
else:
    reference.compare(results, ref)

In [ ]:
# ==== Зафиксировать эталон (выполнять один раз, затем снова закомментировать) ====
# start_run_log(paths, label='freeze')
# from moscow_landcover import reference
# reference.freeze_reference(results, reference.default_reference_path(paths))

## Диагностика (F1/F2) — раскомментировать при перепроверке

In [ ]:
# start_run_log(paths, label='diagnostics')
# from moscow_landcover.diagnostics import availability, run_diagnostics
# f1 = availability(cfg)
# f2 = run_diagnostics(cfg, f1)

## Карта и визуализация

In [ ]:
start_run_log(paths, label='map')

from moscow_landcover import boundaries as boundaries_module
from moscow_landcover.viz import (
    add_boundary_layer, add_classification_layer, add_legend, create_base_map,
)

year = cfg.analysis.historical_years[-1]
territory = 'Historical'
boundaries, areas = boundaries_module.load_boundaries(paths)
result = results[territory][year]

m = create_base_map(territory, cfg.map)
add_boundary_layer(m, boundaries[territory])
add_classification_layer(m, result['classified'], year, analysis=cfg.analysis,
                         map_cfg=cfg.map)
add_legend(m, 'classification', cfg.map)
m

## Экспорт классификаций — раскомментировать при необходимости

In [ ]:
# start_run_log(paths, label='export')
# from moscow_landcover.export import export_classifications_to_drive
# export_classifications_to_drive(
#     results, boundaries,
#     drive_folder='Moscow_LandCover_Classifications',
# )